# Nokken systeem 28 - algoritme motion law

Deze notebook gebruikt het algoritme uit `Algoritme-nok-ontwerp.ipynb`, maar ingevuld voor jouw nokspecificaties.

Voorlopig gaat deze notebook alleen tot en met:

```python
# Plot the motion law
plotMotionLaw(theta_deg, lift, vel_deg, acc_deg)

plt.show()
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Parameters van jouw nok

Motion law keuze:

- `1`: dwell / stilstand
- `5`: 5de-graads veelterm

Specificaties:

- `20° -> 90°`: van `0 mm` naar `25 mm`
- `90° -> 155°`: van `25 mm` naar `10 mm`
- `155° -> 165°`: stilstand op `10 mm`
- `165° -> 200°`: van `10 mm` naar `0 mm`


In [ ]:
# Segment 1: +25 mm tussen 20 deg en 90 deg
startangle1 = 20
endangle1 = 90
theta_seg1 = endangle1 - startangle1
startlift1 = 0
endlift1 = 25
motionlaw1 = 5  # 5th degree polynomial

# Segment 2: -15 mm tussen 90 deg en 155 deg
startangle2 = 90
endangle2 = 155
theta_seg2 = endangle2 - startangle2
startlift2 = 25
endlift2 = 10
motionlaw2 = 5  # 5th degree polynomial

# Segment 3: stilstand tussen 155 deg en 165 deg
startangle3 = 155
endangle3 = 165
theta_seg3 = endangle3 - startangle3
startlift3 = 10
endlift3 = 10
motionlaw3 = 1  # dwell

# Segment 4: -10 mm tussen 165 deg en 200 deg
startangle4 = 165
endangle4 = 200
theta_seg4 = endangle4 - startangle4
startlift4 = 10
endlift4 = 0
motionlaw4 = 5  # 5th degree polynomial


## Arrays initialiseren

`dtheta` is de resolutie van de simulatie in graden.


In [ ]:
dtheta = 0.01  # resolution of simulation, in degrees

theta_deg = np.arange(0, 360, dtheta)  # array of cam angles in degrees
theta = theta_deg * np.pi / 180        # array of cam angles in radians

lift = np.zeros(len(theta))     # lift follower in mm
vel = np.zeros(len(theta))      # velocity follower in mm/rad
vel_deg = np.zeros(len(theta))  # velocity follower in mm/degree
acc = np.zeros(len(theta))      # acceleration follower in mm/rad^2
acc_deg = np.zeros(len(theta))  # acceleration follower in mm/degree^2


## Algoritme: motion segment toevoegen

Dit is dezelfde logica als in de algoritme-notebook, maar hier houden we enkel de motion laws bij die nu nodig zijn: dwell en 5de-graads.


In [ ]:
def addMotionSegment(startangle, endangle, startlift, endlift, motionlaw, dtheta):
    start_index = int(startangle / dtheta)
    end_index = int(endangle / dtheta)
    theta_segment = theta_deg[start_index:end_index]
    beta = endangle - startangle
    x = (theta_segment - startangle) / beta
    L0 = startlift
    L1 = endlift
    L = L1 - L0

    if motionlaw == 1:  # dwell
        lift[start_index:end_index] = L0 * np.ones_like(x)
        vel_deg[start_index:end_index] = np.zeros_like(x)
        acc_deg[start_index:end_index] = np.zeros_like(x)

    elif motionlaw == 5:  # 5th degree poly
        lift[start_index:end_index] = L0 + L * (6 * x**5 - 15 * x**4 + 10 * x**3)
        vel_deg[start_index:end_index] = L / beta * (30 * x**4 - 60 * x**3 + 30 * x**2)
        acc_deg[start_index:end_index] = L / beta**2 * (120 * x**3 - 180 * x**2 + 60 * x)

    else:
        raise ValueError("Deze notebook gebruikt voorlopig alleen motionlaw 1 en 5.")

    vel = vel_deg * 180 / np.pi
    acc = acc_deg * (180 / np.pi)**2

    return lift, vel_deg, acc_deg, vel, acc


## Plotfunctie voor de motion law


In [ ]:
def plotMotionLaw(t, lift, vel, acc):
    fig1, ax1 = plt.subplots(3, 1, figsize=(8, 8), sharex=True)

    ax1[0].plot(t, lift)
    ax1[0].set_ylabel('lift [mm]')
    ax1[0].grid(True)

    ax1[1].plot(t, vel)
    ax1[1].set_ylabel('velocity [mm/deg]')
    ax1[1].grid(True)

    ax1[2].plot(t, acc)
    ax1[2].set_ylabel('acceleration [mm/deg^2]')
    ax1[2].set_xlabel('cam angle [deg]')
    ax1[2].grid(True)

    fig1.tight_layout()


## Segmenten toevoegen


In [ ]:
# Add all four segments to the motion law:
lift, vel_deg, acc_deg, vel, acc = addMotionSegment(startangle1, endangle1, startlift1, endlift1, motionlaw1, dtheta)
lift, vel_deg, acc_deg, vel, acc = addMotionSegment(startangle2, endangle2, startlift2, endlift2, motionlaw2, dtheta)
lift, vel_deg, acc_deg, vel, acc = addMotionSegment(startangle3, endangle3, startlift3, endlift3, motionlaw3, dtheta)
lift, vel_deg, acc_deg, vel, acc = addMotionSegment(startangle4, endangle4, startlift4, endlift4, motionlaw4, dtheta)


## Plot the motion law


In [ ]:
# Plot the motion law
plotMotionLaw(theta_deg, lift, vel_deg, acc_deg)

plt.show()
